# UC-06 - Exploration des sources TPCx-BB
Kevin GREGOIRE - IPSSI - Jour 3

Objectif : vérifier les deux fichiers initiaux complets avant nettoyage. Les chemins HDFS et empreintes sont externalisés. Le refresh produit est inventorié séparément et n'est pas intégré. Contrôles exhaustifs ; seuls les affichages show sont des échantillons.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, '/work/scripts')
from uc06 import session, read_source, profile, save, ROOT, F
config = json.loads((ROOT/'configuration/parametres/execution.json').read_text())
spark = session('UC06-exploration')
print('Spark', spark.version, 'Python', sys.version.split()[0])
print('Mode', spark.sparkContext.master, 'Lot', config['run_id'])

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 11:47:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.7 Python 3.12.3
Mode local[2] Lot 20261118T113628Z-243a5fb5-r3


La session utilise deux threads locaux dans Docker ; les fichiers sont lus dans HDFS. Ce mode ne constitue pas un cluster Spark distribué.

## Source : store_sales
Lecture sans en-tête, séparateur |, schéma ordinal Annexe G. Les conversions sont contrôlées sur toutes les valeurs non nulles.

In [2]:
store_sales, failures_store_sales = read_source(spark, 'store_sales', config['sources']['store_sales']['hdfs'])
store_sales = store_sales.cache()
store_sales.printSchema()
store_sales.select(['ss_ticket_number', 'ss_item_sk', 'ss_quantity', 'ss_sales_price', 'ss_net_paid', 'ss_net_profit']).show(5, truncate=22)
p_store_sales = profile(store_sales, 'store_sales')
print(json.dumps({k:v for k,v in p_store_sales.items() if k != 'schema'}, indent=2, default=str, ensure_ascii=False))
assert not any(failures_store_sales.values())
save('preuves/controles/J3-P04-store_sales.json', p_store_sales)

root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid: decimal(7,2) (nullable = 

+----------------+----------+-----------+--------------+-----------+-------------+
|ss_ticket_number|ss_item_sk|ss_quantity|ss_sales_price|ss_net_paid|ss_net_profit|
+----------------+----------+-----------+--------------+-----------+-------------+
|               0|     10001|         31|         23.08|     715.48|       157.48|
|               0|     10000|         54|         83.34|    4500.36|      -856.98|
|               0|     15556|         39|         83.24|    2629.55|      -470.56|
|               1|      4597|         20|        133.40|    2668.00|      1180.40|
|               1|      3258|         76|         11.85|     900.60|       153.52|
+----------------+----------+-----------+--------------+-----------+-------------+
only showing top 5 rows



26/11/18 11:47:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


{
  "rows": 667579,
  "nulls": {
    "ss_sold_date_sk": 0,
    "ss_sold_time_sk": 0,
    "ss_item_sk": 0,
    "ss_customer_sk": 0,
    "ss_cdemo_sk": 238,
    "ss_hdemo_sk": 295,
    "ss_addr_sk": 448,
    "ss_store_sk": 0,
    "ss_promo_sk": 158,
    "ss_ticket_number": 0,
    "ss_quantity": 0,
    "ss_wholesale_cost": 0,
    "ss_list_price": 0,
    "ss_sales_price": 0,
    "ss_ext_discount_amt": 0,
    "ss_ext_sales_price": 0,
    "ss_ext_wholesale_cost": 0,
    "ss_ext_list_price": 0,
    "ss_ext_tax": 0,
    "ss_coupon_amt": 0,
    "ss_net_paid": 0,
    "ss_net_paid_inc_tax": 0,
    "ss_net_profit": 0
  },
  "numeric": [
    "ss_sold_date_sk",
    "ss_sold_time_sk",
    "ss_item_sk",
    "ss_customer_sk",
    "ss_cdemo_sk",
    "ss_hdemo_sk",
    "ss_addr_sk",
    "ss_store_sk",
    "ss_promo_sk",
    "ss_ticket_number",
    "ss_quantity",
    "ss_wholesale_cost",
    "ss_list_price",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_ext_sales_price",
    "ss_ext_wholesale_c

Les résultats ci-dessus portent sur tout le fichier. Les clés à zéro sont présentes dans le kit et ne sont pas invalidées. Les valeurs nulles ne sont pas remplacées par zéro. Les colonnes *_sk de vente sont des références numériques, pas des dates calendaires.

## Source : item
Lecture sans en-tête, séparateur |, schéma ordinal Annexe G. Les conversions sont contrôlées sur toutes les valeurs non nulles.

In [3]:
item, failures_item = read_source(spark, 'item', config['sources']['item']['hdfs'])
item = item.cache()
item.printSchema()
item.select(['i_item_sk', 'i_product_name', 'i_category', 'i_rec_start_date', 'i_rec_end_date']).show(5, truncate=22)
p_item = profile(item, 'item')
print(json.dumps({k:v for k,v in p_item.items() if k != 'schema'}, indent=2, default=str, ensure_ascii=False))
assert not any(failures_item.values())
save('preuves/controles/J3-P04-item.json', p_item)

root
 |-- i_item_sk: long (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: string (nullable = true)
 |-- i_rec_end_date: string (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: decimal(7,2) (nullable = true)
 |-- i_wholesale_cost: decimal(7,2) (nullable = true)
 |-- i_brand_id: integer (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: integer (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: integer (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: integer (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: integer (nullable = true)
 |-- i_product_name: string (nullable = true)



+---------+----------------------+-----------------+----------------+--------------+
|i_item_sk|        i_product_name|       i_category|i_rec_start_date|i_rec_end_date|
+---------+----------------------+-----------------+----------------+--------------+
|        0|8m9n5Q7T33DNWidoA6n...|Sports & Outdoors|      2000-01-14|          NULL|
|        1|CPB648WB37uUv5OCqjz...|            Books|      2000-09-23|          NULL|
|        2|bMcWnXWH4yWhqADKdlZ...|            Books|      2000-09-01|          NULL|
|        3|           gqbhNi4JCS7|            Books|      2000-12-12|          NULL|
|        4|                  WinA|            Books|      2000-04-04|          NULL|
+---------+----------------------+-----------------+----------------+--------------+
only showing top 5 rows



{
  "rows": 17820,
  "nulls": {
    "i_item_sk": 0,
    "i_item_id": 0,
    "i_rec_start_date": 0,
    "i_rec_end_date": 17820,
    "i_item_desc": 4,
    "i_current_price": 0,
    "i_wholesale_cost": 0,
    "i_brand_id": 8,
    "i_brand": 6,
    "i_class_id": 0,
    "i_class": 0,
    "i_category_id": 0,
    "i_category": 0,
    "i_manufact_id": 0,
    "i_manufact": 0,
    "i_size": 0,
    "i_formulation": 0,
    "i_color": 0,
    "i_units": 0,
    "i_container": 0,
    "i_manager_id": 0,
    "i_product_name": 0
  },
  "numeric": [
    "i_item_sk",
    "i_current_price",
    "i_wholesale_cost",
    "i_brand_id",
    "i_class_id",
    "i_category_id",
    "i_manufact_id",
    "i_manager_id"
  ],
  "text": [
    "i_item_id",
    "i_rec_start_date",
    "i_rec_end_date",
    "i_item_desc",
    "i_brand",
    "i_class",
    "i_category",
    "i_manufact",
    "i_size",
    "i_formulation",
    "i_color",
    "i_units",
    "i_container",
    "i_product_name"
  ],
  "temporal": [
    "i_rec_

Les résultats ci-dessus portent sur tout le fichier. Les clés à zéro sont présentes dans le kit et ne sont pas invalidées. Les valeurs nulles ne sont pas remplacées par zéro. Les colonnes *_sk de vente sont des références numériques, pas des dates calendaires.

## Relation produit et décisions préparatoires
Vérifier l’unicité du référentiel, les références sans correspondance et le maintien du grain vente.

In [4]:
key_errors = item.filter(F.col('i_item_sk').isNull() | (F.col('i_item_sk') < 0)).count()
duplicate_keys = item.count() - item.select('i_item_sk').distinct().count()
unmatched = store_sales.join(item, store_sales.ss_item_sk == item.i_item_sk, 'left_anti').count()
joined = store_sales.join(item.select('i_item_sk','i_product_name'), store_sales.ss_item_sk == item.i_item_sk, 'left')
relation = dict(key_errors=key_errors, duplicate_keys=duplicate_keys, unmatched=unmatched,
                before=store_sales.count(), after=joined.count())
assert key_errors == duplicate_keys == unmatched == 0
assert relation['before'] == relation['after']
print(relation)
save('preuves/controles/J3-P05-jointure-exploration.json', relation)

{'key_errors': 0, 'duplicate_keys': 0, 'unmatched': 0, 'before': 667579, 'after': 667579}


La jointure initiale est vérifiée par la clé de substitution produit, avec un seul produit pour chaque clé. Le couple ticket-produit n'est pas une clé unique de ligne. Aucun dédoublonnage des ventes n'est justifié. Les résultats signés restent conservés. Les dates de fin produit nulles restent nulles ; leur absence n'est pas une erreur de conversion.

In [5]:
print('Bilan :', p_store_sales['rows'], 'ventes ;', p_item['rows'], 'produits')
print('Nulls ventes :', {k:v for k,v in p_store_sales['nulls'].items() if v})
print('Valeurs signées :', p_store_sales['signed_values'])
print('Répétitions ticket-produit :', p_store_sales['ticket_item_repetitions'])
store_sales.unpersist(); item.unpersist(); spark.stop()

Bilan : 667579 ventes ; 17820 produits
Nulls ventes : {'ss_cdemo_sk': 238, 'ss_hdemo_sk': 295, 'ss_addr_sk': 448, 'ss_promo_sk': 158}
Valeurs signées : {'ss_quantity': 0, 'ss_net_paid': 0, 'ss_net_profit': 495971}
Répétitions ticket-produit : 4136


## Conclusion
L'exploration permet de définir les règles avant application : typer les montants en décimaux, convertir les dates produit, retirer seulement les espaces périphériques et conserver les nulls facultatifs. Ne pas exclure les profits négatifs ni les répétitions ticket-produit. Les traitements Silver et Gold sont distincts de ce notebook. Les chiffres de contrôle sont sauvegardés dans preuves/controles ; le rapport documente leurs limites.